![](https://mcd.unison.mx/wp-content/themes/awaken/img/logo_mcd.png)

# Entrenamiento y registr de un modelo en MLflow 3.X usando `pyfunc`

## Aprendizaje Automático Aplicado

### Maestría en Ciencia de Datos

#### **Julio Waissman**, 2026

[**Abrir en google Colab**](https://colab.research.google.com/github/juliowaissman/mlflow2X-serve-ejemplo/blob/main/entrenamiento.ipynb)


## 0. Inicialización de librerías y entorno

In [ ]:
# Si no está instalado, hay que instalar MLflow
!pip install --upgrade "mlflow>=3.1"

Recuerda que para que funcione tienes que habilitar los siguientes secretos en Colab:

- `DATABRICKS_HOST`
- `DATABRICKS_TOKEN`
- `MLFLOW_EXPERIMENT_ID`
- `MLFLOW_TRACKING_URI`

Si lo ejecutas en forma local, necesitas esa información en un archivo .env, que no debes compartir en tu repositorio

Para saber como hacer esto, te recmiendo revises [esta libreta](https://colab.research.google.com/github/mcd-unison/aaa-curso/blob/main/ejemplos/mlflow3-et.ipynb)

In [ ]:
import mlflow
import mlflow.pyfunc
import mlflow.models.signature

import pandas as pd
from typing import Any
import numpy as np
import pickle

from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from google.colab import userdata
import os

In [ ]:
DATABRICKS_TOKEN = userdata.get('DATABRICKS_TOKEN')
DATABRICKS_HOST = userdata.get('DATABRICKS_HOST')
MLFLOW_TRACKING_URI = userdata.get('MLFLOW_TRACKING_URI')
MLFLOW_EXPERIMENT_ID = userdata.get('MLFLOW_EXPERIMENT_ID')

os.environ['DATABRICKS_TOKEN'] = DATABRICKS_TOKEN
os.environ['DATABRICKS_HOST'] = DATABRICKS_HOST
os.environ['MLFLOW_TRACKING_URI'] = MLFLOW_TRACKING_URI
os.environ['MLFLOW_EXPERIMENT_ID'] = MLFLOW_EXPERIMENT_ID

Y ahora vamos a preguntarle a Databricks algunas cosas para establecer como guardar el modelo. Primero veamos cuales experimentos ya tenemos registrados:

In [ ]:
experiments = mlflow.search_experiments()
for exp in experiments:
    print(f"Experiment ID: {exp.experiment_id}, Name: {exp.name}")

Y ahora completa esta celda seleccionando el experimento (si fuera necesario):

In [ ]:
# Si ya lo tienes no necesitas ejecutar esta celda
os.environ['MLFLOW_EXPERIMENT_ID'] = XXXXXXXXXX # <- Aquí pon el nñumero de experimento

Para guardar artefactos y modelos ahora hay que especificar el catálogo y el esquema donde se van a guardar para el Unity catalog que permite mantener la gobernanza de datos y modelos.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient(host=DATABRICKS_HOST, token=DATABRICKS_TOKEN)

print("Available Catalogs and Schemas:")
for catalog in w.catalogs.list():
    print(f"  Catalog: {catalog.name}")
    for schema in w.schemas.list(catalog_name=catalog.name):
        print(f"    Schema: {schema.name}")

## 1. Definir un modelo usando `pyfunc`

Una forma flexible de definir modelos que se van a poner en producción en MLflow es el uso de la clase `PythonModel`de `mlflow.pyfunc`. Esta clase nos permite definir un modelo personalizado, con su propia lógica de inferencia, y luego registrarlo en MLflow para su posterior uso en producción.

El metodo `load_context(self, context)` es obligatorio. En este caso, vamos a asumir que tenemos un modelo preentrenado guardado en forma de `pickle` en los artefactos, como `rf_model`. Así que en la función contexto será lo único que cargemos. Sin embargo, en el contexto podríamos cargar diferentes modelos, provenientes de diferentes librerías, o funciones de preprocesamiento e inferencia muy personalizadas.

Asumiento que el contexto se carga de inicio, el método `predict(self, context, model_input)` es el que se va a llamar cada vez que se haga una petición de inferencia al modelo. En este método, podemos definir la lógica de inferencia personalizada que queremos aplicar a los datos de entrada. En nuestro caso, al ser muy simple y estar usando un modelo de `sklearn`, simplemente vamos a llamar al método `predict` del modelo cargado en el contexto.

In [ ]:
class CustomIrisModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        # Cargar el modelo base desde los artefactos
        with open(context.artifacts["rf_model"], "rb") as f:
            self.model = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> np.ndarray:
        # Aquí puedes poner lógica personalizada (ej. validación, limpieza)
        # Convertimos forzosamente a float para demostrar el preprocesamiento
        #
        # Es importqnte especificar en este método el tipo de dato 
        # de entrada y salida, para que MLflow pueda generar la firma del modelo
        processed_input = model_input.astype(float)
        return self.model.predict(processed_input)

## 2. Carga de datos

Esta etapa es muy estandar. Vamos a cargar el conjunto de iris porque es un ejemplo de chocolate para mostrar como puede funcionar.

In [ ]:
data = load_iris()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## 3. Entrenamiento y registro del modelo

En esta etapa, vamos a entrenar un modelo de Random Forest con diferentes hiperparámetros y luego registrarlo en MLflow usando la clase `PythonModel` que definimos anteriormente. 

Cuando realizamos el registro de experimentos en MLflow, cada ejecución se registra como un "run" dentro de un experimento. Cada run tiene un nombre que podemos asignar para identificarlo fácilmente. En este caso, estamos usando el parámetro `run_name` para darle un nombre descriptivo a cada ejecución, lo que nos ayudará a diferenciarlos cuando los veamos en la interfaz de MLflow.

A diferencia de solo registrar los experimentos, aqui amos a registrar el modelo completo, con su lógica de inferencia personalizada, lo que nos permitirá luego usarlo directamente en producción sin necesidad de reentrenar o redefinir la lógica de inferencia. Para eso usamos el método `mlflow.pyfunc.log_model`, que nos permite registrar un modelo definido como una clase `PythonModel` junto con sus dependencias y artefactos necesarios para su funcionamiento.

En este caso no es necesario explicitar las entradas y salidas del modelo, porque al ser un modelo de `sklearn` es muy sencillo, pero en casos más complejos podríamos querer definir explícitamente las entradas y salidas del modelo para facilitar su uso en producción. Para eso, podríamos usar el parámetro `input_example` para proporcionar un ejemplo de entrada al modelo, lo que ayudaría a entender qué tipo de datos espera el modelo y cómo formatearlos correctamente para su uso en producción.

In [ ]:
# Aqui se pone catalogo.schema.nombre_modelo
MODEL_NAME = "workspace.default.Iris_RF"

def train_and_log_model(n_estim, max_d, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        # Entrenar modelo
        clf = RandomForestClassifier(
            n_estimators=n_estim,
            max_depth=max_d,
            random_state=42
        )
        clf.fit(X_train, y_train)

        # Evaluar
        preds = clf.predict(X_test)
        acc = accuracy_score(y_test, preds)

        # Guardar el modelo localmente de forma temporal
        model_path = "temp_rf_model.pkl"
        with open(model_path, "wb") as f:
            pickle.dump(clf, f)

        # Inferir la firma del modelo explícitamente
        signature = mlflow.models.signature.infer_signature(
            X_train, 
            clf.predict(X_train)
        )

        # Registrar parámetros y métricas
        mlflow.log_params({"n_estimators": n_estim, "max_depth": max_d})
        mlflow.log_metric("accuracy", acc)

        # Empaquetar y registrar usando PyFunc
        artifacts = {"rf_model": model_path}

        mlflow.pyfunc.log_model(
            artifact_path="model",
            python_model=CustomIrisModel(),
            artifacts=artifacts,
            registered_model_name=MODEL_NAME,
            signature=signature, # Pass the explicitly inferred signature
            input_example=X_train.sample(1)
        )
        print(f"[{run_name}] Modelo registrado con Accuracy: {acc:.4f}")
        os.remove(model_path) # Limpiar temporal

## 4. Entrenamiento y registro de dos modelos

Vamos a registrar dos modelos para probar como funciona lo de las etiquetas de modelo en MLflow. 

Un modelo va a ser claramente más malo que el otro para ver como se pueden asignar etiquetas, pero eso ya es en otra parte del tutorial.

In [ ]:
mlflow.set_registry_uri("databricks-uc")
train_and_log_model(n_estim=10, max_d=2, run_name="Modelo_Base")
train_and_log_model(n_estim=50, max_d=5, run_name="Modelo_Mejorado")